<a href="https://colab.research.google.com/github/kyle-g-stubblefield/think-python/blob/main/game_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Code for our python mud

it uses diku color codes

In [ ]:
"""
diku_color.py
─────────────
Converts Diku/ROM/SMAUG-style color codes to ANSI escape sequences,
with a Colab-aware HTML renderer that matches the classic xterm 16-color
dark-terminal palette exactly.

Usage:
    cprint("&+cAshenmoor&N -- &Rwarning&N!")

    In a real terminal  → ANSI escape codes
    In Colab / Jupyter  → HTML with CSS colors on a black background,
                          matching xterm/PuTTY dark-terminal colors

Color codes:
    &N / &n   reset
    &r / &R   dark red      / bright red
    &g / &G   dark green    / bright green
    &b / &B   dark blue     / bright blue
    &c / &C   dark cyan     / bright cyan
    &y / &Y   dark yellow   / bright yellow
    &m / &M   dark magenta  / bright magenta
    &w / &W   dark grey     / bright white
    &x / &X   black         / dark grey
    &+X       same as uppercase (explicit bright-bit prefix)
    &&        literal ampersand
"""

import re

# ── ANSI escape helpers ───────────────────────────────────────────────────────

RESET = "\033[0m"

def _ansi(code: int, bold: bool = False) -> str:
    return f"\033[1;{code}m" if bold else f"\033[{code}m"

_BARE = {
    "x": (30, False),  "X": (30, True),
    "r": (31, False),  "R": (31, True),
    "g": (32, False),  "G": (32, True),
    "y": (33, False),  "Y": (33, True),
    "b": (34, False),  "B": (34, True),
    "m": (35, False),  "M": (35, True),
    "c": (36, False),  "C": (36, True),
    "w": (37, False),  "W": (37, True),
}
_PLUS = {ch.lower(): (code, True) for ch, (code, _) in _BARE.items()}
_PLUS.update({ch.upper(): (code, True) for ch, (code, _) in _BARE.items()})


# ── Classic xterm 16-color palette (dark background) ─────────────────────────
#
# These are the standard hex values used by xterm, PuTTY, and most
# Linux terminal emulators. They're what players would have seen on a
# real Diku MUD in the 90s. Colab dark mode background is very close
# to #0d1117 so these read almost identically to a real terminal.
#
#   Index  ANSI  Color            Normal      Bright
#     0    30    black            #000000     #555555
#     1    31    red              #AA0000     #FF5555
#     2    32    green            #00AA00     #55FF55
#     3    33    yellow           #AA5500     #FFFF55
#     4    34    blue             #0000AA     #5555FF
#     5    35    magenta          #AA00AA     #FF55FF
#     6    36    cyan             #00AAAA     #55FFFF
#     7    37    white            #AAAAAA     #FFFFFF

_XTERM_PALETTE = {
    #        (ansi_code, bold)  ->  hex color
    (30, False): "#000000",   # black
    (30, True):  "#555555",   # dark grey
    (31, False): "#AA0000",   # dark red
    (31, True):  "#FF5555",   # bright red
    (32, False): "#00AA00",   # dark green
    (32, True):  "#55FF55",   # bright green
    (33, False): "#AA5500",   # dark yellow / olive
    (33, True):  "#FFFF55",   # bright yellow
    (34, False): "#0000AA",   # dark blue
    (34, True):  "#5555FF",   # bright blue
    (35, False): "#AA00AA",   # dark magenta
    (35, True):  "#FF55FF",   # bright magenta
    (36, False): "#00AAAA",   # dark cyan
    (36, True):  "#55FFFF",   # bright cyan
    (37, False): "#AAAAAA",   # dark white / grey
    (37, True):  "#FFFFFF",   # bright white
}


# ── Token parser (shared by both renderers) ───────────────────────────────────

def _tokenize(text: str):
    """
    Yield (kind, value) tuples:
        ("text",   str)
        ("color",  (ansi_code, bold))
        ("reset",  None)
    """
    i = 0
    n = len(text)
    while i < n:
        if text[i] != "&":
            # collect a run of plain characters
            j = i + 1
            while j < n and text[j] != "&":
                j += 1
            yield ("text", text[i:j])
            i = j
            continue

        if i + 1 >= n:
            yield ("text", "&"); i += 1; continue

        ch = text[i + 1]

        if ch == "&":
            yield ("text", "&"); i += 2; continue

        if ch in ("N", "n"):
            yield ("reset", None); i += 2; continue

        if ch == "+" and i + 2 < n:
            entry = _PLUS.get(text[i + 2])
            if entry:
                yield ("color", entry); i += 3; continue
            yield ("text", "&"); i += 1; continue

        entry = _BARE.get(ch)
        if entry:
            yield ("color", entry); i += 2; continue

        yield ("text", "&"); i += 1


# ── ANSI renderer ─────────────────────────────────────────────────────────────

def diku_to_ansi(text: str) -> str:
    """Replace Diku color codes with ANSI escape sequences."""
    parts = []
    for kind, value in _tokenize(text):
        if kind == "text":
            parts.append(value)
        elif kind == "color":
            parts.append(_ansi(*value))
        elif kind == "reset":
            parts.append(RESET)
    return "".join(parts)


# ── HTML renderer (Colab / Jupyter) ──────────────────────────────────────────

def _html_escape(s: str) -> str:
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def diku_to_html(text: str, bg: str = "#0d0d0d") -> str:
    """
    Convert Diku color codes to an HTML snippet styled to match a classic
    dark-background terminal using the standard xterm 16-color palette.

    bg defaults to near-black (#0d0d0d), matching Colab dark mode.
    Pass bg=None to omit the wrapper div (embed in your own container).
    """
    DEFAULT_FG = "#AAAAAA"   # grey — same as a bare terminal prompt

    parts = []
    current_color = None    # None means "use default fg"

    def open_span(color_hex):
        return f'<span style="color:{color_hex}">'

    for kind, value in _tokenize(text):
        if kind == "text":
            txt = _html_escape(value).replace("\n", "<br>")
            if txt:
                if current_color:
                    parts.append(open_span(current_color))
                    parts.append(txt)
                    parts.append("</span>")
                else:
                    parts.append(f'<span style="color:{DEFAULT_FG}">{txt}</span>')

        elif kind == "color":
            current_color = _XTERM_PALETTE.get(value, DEFAULT_FG)

        elif kind == "reset":
            current_color = None

    inner = "".join(parts)

    if bg is None:
        return inner

    return (
        f'<div style="background:{bg}; font-family:\'Courier New\',monospace; '
        f'font-size:14px; padding:8px 12px; border-radius:4px; '
        f'line-height:1.5; white-space:pre-wrap;">'
        f'{inner}</div>'
    )


# ── cprint: auto-detects Colab vs real terminal ───────────────────────────────

def _in_notebook() -> bool:
    """True when running inside Jupyter / Colab."""
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False


def cprint(*args, sep: str = " ", end: str = "\n") -> None:
    """
    Drop-in replacement for print() that colorizes Diku codes.

    Accepts any objects, exactly like print() — each argument is converted
    to a string via str(), joined with sep, then colorized for the current
    environment:

        Terminal   : ANSI escape codes
        Colab/Jupyter : HTML rendered via IPython.display

    This means __repr__ / __str__ methods can return raw Diku-coded strings
    and cprint() will handle the colorization automatically:

        class Room:
            def __str__(self):
                return "&+WTHE RUSTY FLAGON&N\\nSmoky and warm."

        cprint(room)          # works — no color() call needed inside __str__
        cprint(room, player)  # works — stringifies both, joins, colorizes
        cprint("&Rhp:&N", 42) # works — mixed types fine
    """
    text = sep.join(str(a) for a in args)
    if _in_notebook():
        from IPython.display import display, HTML
        display(HTML(diku_to_html(text)))
    else:
        print(diku_to_ansi(text) + RESET, end=end)


def cstrip(text: str) -> str:
    """Remove all Diku color codes, returning plain text."""
    return re.sub(r"&&|&[Nn]|&\+?[a-zA-Z]",
                  lambda m: "&" if m.group() == "&&" else "",
                  text)

def color(text: str) -> str:
  return diku_to_ansi(text) + RESET

class ColorString(str):
    """
    A str subclass returned by color(..., "html").

    Behaves exactly like a regular string in every context — concatenation,
    slicing, len(), storing in a variable — but adds two extras:

      _repr_html_()   Colab/Jupyter calls this automatically when the object
                      is the last expression in a cell, so it renders as
                      colored HTML without you doing anything special.

      .display()      Explicitly push the HTML to the Colab output cell.
                      Use this when the color() call is NOT the last
                      expression, e.g. inside a loop or function.

    Never pass to print() — print() always writes the raw string, bypassing
    the HTML renderer.  Use cprint() or .display() instead.
    """
    def _repr_html_(self):
        return str(self)

    def display(self):
        """Render this HTML string in the current Colab / Jupyter output cell."""
        from IPython.display import display as _display, HTML
        _display(HTML(str(self)))


def color(text: str, mode: str = "ansi") -> str:
    """
    Convert a Diku-colored string and return the result.

    Parameters
    ----------
    text : str
        String containing Diku color codes (&+c, &R, &N, etc.)
    mode : str
        "ansi"  (default)
            Returns a plain str with ANSI escape sequences.
            Correct output when passed to print() in a real terminal.

        "html"
            Returns a ColorString (str subclass) containing an HTML
            snippet with inline CSS colors on a dark background.

            DO NOT pass to print() — print() always writes raw text.
            Instead use one of:
              color(s, "html").display()        explicit render
              cprint(s)                         auto-detects env, always right
            Or let it be the last expression in a Colab cell and Colab
            renders it automatically via _repr_html_().

    Returns
    -------
    str or ColorString
        "ansi" mode  ->  plain str  (ANSI escapes inside)
        "html" mode  ->  ColorString  (HTML that auto-renders in Colab)

    Examples
    --------
        # terminal
        print(color("&+cHello&N world"))

        # Colab — three equivalent ways
        cprint("&+cHello&N world")                    # easiest, always correct
        color("&+cHello&N world", "html").display()   # explicit render
        color("&+cHello&N world", "html")             # last expression in cell
    """
    if mode == "html":
        return ColorString(diku_to_html(text))
    return diku_to_ansi(text) + RESET

In [38]:
def cinput(prompt: str = "") -> str:
    """
    Colorized input(). Displays a Diku-coded prompt then reads a line.

    Terminal  : passes the ANSI-converted prompt directly to input(), so
                the prompt and cursor appear on the same line as normal.
    Colab     : cprints the prompt (rendered HTML), then calls input("")
                so the text box appears on the line below the prompt.

    Returns the raw string the user typed, stripped of leading/trailing
    whitespace. Never returns color codes — input is always plain text.

    Examples
    --------
        name = cinput("&+WEnter your name:&N ")
        cmd  = cinput("&Y>&N ")
    """
    if _in_notebook():
        cprint(prompt, end="")
        return input("").strip()
    else:
        return input(diku_to_ansi(prompt) + RESET).strip()

def crepl(
    handler,
    prompt:    str = "&g> &N ",
    quit_cmds: tuple = ("quit", "exit", "q"),
    banner:    str = "",
    farewell:  str = "",
) -> None:
    """
    A Diku-color-aware REPL loop.

    Displays a colored prompt, reads a line of input, passes it to
    `handler`, and cprints whatever the handler returns.  Repeats until
    the user types a quit command or EOF (Ctrl-D / Ctrl-C).

    Parameters
    ----------
    handler : callable(str) -> object
        Called with the raw input string each iteration.
        The return value is passed to cprint() — any object whose
        __str__ returns a Diku-coded string will render correctly.
        Return None or "" to print nothing for that turn.

    prompt : str
        Diku-coded prompt string shown before each input.
        Default: "&Y>&N "  (yellow > then reset)

    quit_cmds : tuple of str
        Input strings that end the loop (case-insensitive).
        Default: ("quit", "exit", "q")

    banner : str
        Diku-coded string printed once before the loop starts.
        Pass "" to skip.

    farewell : str
        Diku-coded string printed once after the loop ends.
        Pass "" to skip.

    Examples
    --------
        # Minimal — echo input back in cyan
        crepl(lambda s: f"&+c{s}&N")

        # Game loop
        crepl(
            handler   = lambda cmd: game.process(cmd),
            prompt    = "&Y> &N",
            banner    = "&+WASHENMOOR&N  -- type help",
            farewell  = "&+RFarewell.&N",
        )
    """
    if banner:
        cprint(banner)

    while True:
        try:
            raw = cinput(prompt)
        except (EOFError, KeyboardInterrupt):
            break
        if not raw:
            continue
        result = handler(raw)
        if result == 'quit':
            break
        if result is not None and result != "":
            cprint(result)

    if farewell:
        cprint(farewell)

In [36]:
# ── Demo (terminal mode) ──────────────────────────────────────────────────────

if __name__ == "__main__":
    samples = [
        "&+cAshenmoor&N -- a text adventure",
        "&Rwarning:&N health is &+Rcritically low&N!",
        "&+WTHE RUSTY FLAGON&N",
        "&WPlatinum&N: 152 &YGold&N: 42  &wSilver&N: 7  &yCopper&N: 3",
        "&ggreen&N  &+Gbright green&N  &ccyan&N  &+cbright cyan&N",
        "&mmagenta&N  &Mbright magenta&N  &bblue&N  &Bbright blue&N",
        "&&N is a literal ampersand-N, not a reset",
    ]
    print("\n\033[1m-- terminal ANSI demo --\033[0m")
    for s in samples:
        print(f"  raw: {s}")
        cprint(f"  out: {s}")
        print()
    print("\033[1m-- cstrip --\033[0m")
    for s in samples:
        print(f"  {cstrip(s)}")


-- terminal ANSI demo --
  raw: &+cAshenmoor&N -- a text adventure



  raw: &Rwarning:&N health is &+Rcritically low&N!



  raw: &+WTHE RUSTY FLAGON&N



  raw: &WPlatinum&N: 152 &YGold&N: 42  &wSilver&N: 7  &yCopper&N: 3



  raw: &ggreen&N  &+Gbright green&N  &ccyan&N  &+cbright cyan&N



  raw: &mmagenta&N  &Mbright magenta&N  &bblue&N  &Bbright blue&N



  raw: &&N is a literal ampersand-N, not a reset



-- cstrip --
  Ashenmoor -- a text adventure
  THE RUSTY FLAGON
  Platinum: 152 Gold: 42  Silver: 7  Copper: 3
  green  bright green  cyan  bright cyan
  magenta  bright magenta  blue  bright blue
  &N is a literal ampersand-N, not a reset


In [7]:
from enum import Enum

class Stats(Enum):
  STRENGTH = 0
  DEXTERITY = 1
  CONSTITUTION = 2
  INTELLIGENCE = 3
  WISDOM = 4
  CHARISMA = 5

  @property
  def abv(self):
    return self.name.lower()[:3]

In [12]:
## Base Race Class
class Race():
  def __init__(self, d = {
      'stats_modifier': [1, 1, 1, 1, 1, 1],
      'innate_abilities': [],
      'size': 'medium'}, statlist = Stats):
    self.stats_modifiers = d['stats_modifier']

  def get_mod(self, stat):
    if type(stat) == int:
      return self.stats_modifiers[stat]
    elif type(stat) == str:
      for k in Stats:
        if stat == k.abv:
          return self.stats_modifiers[k.value]
    return None

In [8]:
## Base Character Class
class Character():
  """ Base character class """

  def __init__(self, d, statlist = Stats):
    self.name = d.get('name', None)
    self.stats = d.get('stats', [80,80,80,80,80,80])
    self.race = d.get('race', 'Human')
    self.level = d.get('level', 1)
    self.position = d.get('position', 'standing')
    self.cclass = d.get('class', 'Warrior')

  def __str__(self):
    self.pcs()


  def get_stat(self, stat):
    if type(stat) == int:
      return self.stats[stat]
    elif type(stat) == str:
      for k in Stats:
        if stat == k.abv:
          return self.stats[k.value]
    return None


  def computed_stat(self, stat):
    return int(self.get_stat(stat) * races[self.race].get_mod(stat))

  def pcs(self):
    print(f'Character sheet for {self.name}\n')
    print(f'Race: {self.race}')
    print(f'Class: {self.cclass}')
    print(f'Level: {self.level}')
    print(f'Stats:')
    print(f'Strength:     {self.get_stat('str'):>3}    Intelligence: {self.get_stat('int'):>3}')
    print(f'Dexterity:    {self.get_stat('dex'):>3}    Wisdom:       {self.get_stat('wis'):>3}')
    print(f'Constitution: {self.get_stat('con'):>3}    Charisma:     {self.get_stat('cha'):>3}')
    print('\n\n')

In [9]:
class Object():
  def __init__(self, d = {
      'name': None,
      'room_description': None,
      'key_words': (None),
      'description': None}):
    self.name = d['name']
    self.room_description = d['room_description']
    self.key_words = d['key_words']
    self.description = d['description']

In [31]:
terrain = ('no ground', 'water', 'mountain', 'plain', 'stone')

class Room():
  def __init__(self, d):
    self.number = d['number']
    self.name = d['name']
    self.description = d['description']
    self.indoors = d['indoors']
    self.terrain = d['terrain']
    self.exits = d['exits']
    self.objects = d['objects']

  def _exits(self):
    output = "&gExits:&N"
    if len(self.exits) == 0:
      output += " &RNone!&N"
    else:
      count = 0
      for exit in self.exits:
        if count > 0:
          output += "&C -&N"
        output += f'&c {exit['direction'].title()}&N'
        count += 1
    return output + "\n&N"

  def _objects(self):
    output = ""
    if len(self.objects) > 0:
      for obj in self.objects:
        output += f'{obj.room_description}\n'
    return output


  @property
  def characters(self):
    char = []
    for c, r in locations.items():
      if r == self.number:
        char.append(c)
    return char

  def _characters(self):
    output = ""
    if len(self.characters) > 0:
      for character in self.characters:
        output += f'{characters[character].name.title()} stands here\n'
    return output


  def __repr__(self):
    output = f'{self.name}\n     {self.description}\n'
    output += self._exits()
    output += self._objects()
    output += self._characters()
    return output

In [13]:
## Creates Race Instances
#from character import Character
#from race import Race


races = {}
races['Human'] = Race()
races['Dwarf'] = Race({'stats_modifier': [1.1, 1, 1.3, .9, 1, 10], 'size': 'medium_small', 'innate_abilities': ['infravision']})
races['Grey Elf'] = Race({'stats_modifier': [.7, 1.2, .8, 1.1, 1, 1.1], 'size': 'medium_small', 'innate_abilities': ['infravision', 'outdoor_sneak']})
races['Ogre'] = Race({'stats_modifier': [1.5, .8, 1.5, .5, .75, .75], 'size': 'large' ,'innate_abilities': ['doorbash']})

## Create a moted Instance
characters = {}
characters['Moted'] = Character({
    'name': "Moted",
    'race': 'Dwarf',
    'class': 'Shaman',
    'level': 24,
    "stats": [88, 80, 80, 80, 80, 80]})
characters['Aleolas'] = Character({
    'name': "Aleolas",
    'race': 'Grey Elf',
    'class': 'Ranger',
    'level': 50,
    "stats": [100, 80, 100, 80, 80, 80]})
characters['Illilel'] = Character({
    'name': "Illilel",
    'race': "Grey Elf",
    'class': "Bard",
    'level': 50,
    'stats': [100, 100, 100, 72, 54, 100]
})

def character_list(char_list):
  print(f'{"Name":<15} {"Race":<12} {"Class":<10} {"Level":>2}')
  print("_" * 45, end="\n\n")

  for (_, char) in char_list.items():
    print(f'{char.name:<15} {char.race:<12} {char.cclass:<10} {char.level:>5}')

character_list(characters)

Name            Race         Class      Level
_____________________________________________

Moted           Dwarf        Shaman        24
Aleolas         Grey Elf     Ranger        50
Illilel         Grey Elf     Bard          50


In [14]:
print(characters['Moted'].get_stat('con'))
print(characters['Moted'].computed_stat('con'))

80
104


In [34]:
rooms = {}
objects = []

tss = {
    'name': "a &+rtattered &+csilken sack&N",
    'key_words': ('tattered', 'silken', 'sack'),
    'room_description': "a &+rtattered &+csilken sack&N lies here, discarded.",
    'description': "This sack seems in a awful condition, large holes open in its silken\nfabric but strangely enough nothing exits from them."
}
windsong = {
    'name': "&+ga &wg&Wl&wi&Wtt&wer&Wi&wng &N&+gelven scimitar&N",
    'room_description': "&+gA glittering elven scimitar is lying on the ground here.&N",
    'key_words': ('scimitar', 'elven', 'glittering'),
    'description': """&+gIts blade encrusted with diamond dust, this magically light
&+gelven blade glitters in the sunlight and seems to hum softly
&+gwhen wielded in battle.&N"""
}
objects.append(Object(
    {'name': 'red expo marker',
     'key_words': ('red', 'expo', 'marker'),
     'room_description': 'a red expo marker is carlessly discarded here',
     'description': 'Dark magenta low scent marker, half used'}
    ))

objects.append(Object(
    {'name': 'green expo marker',
     'key_words': ('green', 'expo', 'marker'),
     'room_description': 'a green expo marker is carlessly discarded here',
     'description': 'Forest green low scent marker, half used'}))

rooms[1] = Room({
    'number': 1,
    'name': 'The Void',
    'description': 'There is nothing here but the sound of rushing of wind.\nWe are waiting for the Spirit of God to move over it.',
    'indoors': False,
    'terrain': 'no ground',
    'exits': [
        {'direction': "north", "roomId": 1},
        {'direction': "south", "roomId": 1},
        {'direction': "east", "roomId": 2},
        {'direction': "west", "roomId": 3},
        {'direction': "up", "roomId": 1},
        {'direction': "down", "roomId": 1}
        ],
    'objects':[]})

rooms[2] = Room({
    'number': 2,
    'name': 'Not The Void',
    'description': 'You left\nSo sorry',
    'indoors': False,
    'terrain': 'no ground',
    'exits': [
        {'direction': "north", "roomId": 1},
        {'direction': 'south', 'roomId': 1}
        ],
    'objects': [*objects]})

rooms[3] = Room({
    'number': 3,
    'name': 'Not the Not The Void',
    'description': 'Now you stuck',
    'indoors': False,
    'terrain': 'no ground',
    'exits': [],
    'objects': [*objects]})

ptr = Object(tss)
objects.append(ptr)
rooms[2].objects.append(ptr)
ptr = Object(windsong)
objects.append(ptr)
rooms[2].objects.append(ptr)

#print(here)
locations = {'Moted': 1,
    'Illilel': 1,
    'Aleolas': 1}

In [ ]:
for c, r in locations.items():
  print(c)

Moted
Illilel
Aleolas


In [22]:
def go(character, locations, rooms, direction):
  #print(rooms[positions[character]])
  print(direction)
  room = rooms[locations[character]]

  #loop over the exits in the room
  for rm in room.exits:
    if rm['direction'] == direction:
      rmid = rm['roomId']
      cprint(rooms[rmid])
      locations[character] = rmid


      #cuts the loop short if found
      return

  #sanity check, exit not found print alas
  print("Alas, you cannot go that way. . . .")

In [35]:
go('Moted', locations, rooms, 'east')
print(locations)

east


{'Moted': 2, 'Illilel': 1, 'Aleolas': 1}


In [ ]:
## Create a new room instance
## name description and at least 1 exit

## Create a new object instance

## Create a new character instance

In [ ]:
print(objects)

def get_object(key_word, objects):
  for obj in objects:
    if key_word in obj.key_words:
      return obj
  return None

marker = get_object('green', objects)
print(marker)